In [3]:
# pip install python-dotenv


Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import openai
import os
import boto3

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())  # read local .env file

openai.api_key = os.environ["OPENAI_API_KEY"]  # key removed before publishing (2026)
model = 'gpt-3.5-turbo'
openai.api_key = os.environ["OPENAI_API_KEY"]  # key removed before publishing (2026)


In [4]:
s3 = boto3.client('s3')
BUCKET_NAME = 'capstonehaystacks'
FILE_NAME = 'points-of-interest-google2.csv'

file_obj = s3.get_object(Bucket=BUCKET_NAME, Key=FILE_NAME)
df = pd.read_csv(file_obj['Body'])

In [28]:
df['primary_category'].value_counts()

Restaurant                      2374
Gas station                     1965
Auto repair shop                1635
Hair salon                      1529
Nail salon                      1318
                                ... 
Patent attorney                    1
Wine wholesaler and importer       1
Bowling club                       1
Dry wall supply store              1
National reserve                   1
Name: primary_category, Length: 2531, dtype: int64

In [33]:
counts = df['primary_category'].value_counts()

# Filter the counts Series to only include values that occur more than 30 times
filtered_counts = counts[counts > 50]

# Print the filtered counts
print(filtered_counts)


Restaurant                2374
Gas station               1965
Auto repair shop          1635
Hair salon                1529
Nail salon                1318
                          ... 
Engineering consultant      51
Professional services       51
Real estate developer       51
Print shop                  51
Nature preserve             51
Name: primary_category, Length: 346, dtype: int64


In [40]:
df.shape

(94231, 17)

In [36]:
# Check for duplicates in the entire DataFrame
duplicates = df.duplicated()

# Print the number of duplicate rows
print(f"Number of duplicate rows: {duplicates.sum()}")



Number of duplicate rows: 0


In [39]:
# Check for duplicates in a specific column
duplicates = df['name'].duplicated()

# Print the number of duplicate values in the column
print(f"Number of duplicate values: {duplicates.sum()}")

Number of duplicate values: 21827


In [2]:

# Load the CSV file into a DataFrame
# df = pd.read_csv('points-of-interest-google2.csv')

In [10]:
categories_set = {'accounting', 'airport', 'amusement_park', 'aquarium', 'art_gallery', 'atm', 'bakery', 'bank', 'bar', 'beauty_salon', 'bicycle_store', 'book_store', 'bowling_alley', 'bus_station', 'cafe', 'campground', 'car_dealer', 'car_rental', 'car_repair', 'car_wash', 'casino', 'cemetery', 'church', 'city_hall', 'clothing_store', 'convenience_store', 'courthouse', 'dentist', 'department_store', 'doctor', 'drugstore', 'electrician', 'electronics_store', 'embassy', 'fire_station', 'florist', 'funeral_home', 'furniture_store', 'gas_station', 'gym', 'hair_care', 'hardware_store', 'hindu_temple', 'home_goods_store', 'hospital', 'insurance_agency', 'jewelry_store', 'laundry', 'lawyer', 'library', 'light_rail_station', 'liquor_store', 'local_government_office', 'locksmith', 'lodging', 'meal_delivery', 'meal_takeaway', 'mosque', 'movie_rental', 'movie_theater', 'moving_company', 'museum', 'night_club', 'painter', 'park', 'parking', 'pet_store', 'pharmacy', 'physiotherapist', 'plumber', 'police', 'post_office', 'primary_school', 'real_estate_agency', 'restaurant', 'roofing_contractor', 'rv_park', 'school', 'secondary_school', 'shoe_store', 'shopping_mall', 'spa', 'stadium', 'storage', 'store', 'subway_station', 'supermarket', 'synagogue', 'taxi_stand', 'tourist_attraction', 'train_station', 'transit_station', 'travel_agency', 'university', 'veterinary_care', 'zoo'}




In [11]:
print(len(categories_set))

96


In [12]:
businesses_set = set(df['primary_category'])


In [13]:
# converting to a list
businesses_list = list(businesses_set)

# Breaking up the businesses_set so it doesnt exceed token length
businesses1 = set(businesses_list[:500])
businesses2 = set(businesses_list[500:1000])
businesses3 = set(businesses_list[1000:1500])
businesses4 = set(businesses_list[1500:2000])
businesses5 = set(businesses_list[2000:])


In [ ]:
# Get the first 500 elements from primary_category_set
# businesses = set(list(primary_category_set)[:500])

In [10]:
# categories_list = df['primary_category'].tolist()

In [11]:
# categories_set = set(category_list)


In [30]:
# businesses = primary_category_list[:20]

In [52]:
# businesses = primary_category_list[:100]

In [61]:
print(len(businesses_set))

2532


In [7]:
# print(len(primary_category_set))

In [8]:
# primary_category_set = set(primary_category_list)

In [9]:
# businesses_set = set(primary_category_set)

In [11]:
# # Get the first 500 elements from primary_category_set
# businesses = set(list(primary_category_set)[:500])

# # Print the new set
# print(businesses)

In [14]:
# using sets instead of lists

def get_completion(messages, model="gpt-3.5-turbo"):
    response = openai.ChatCompletion.create(
        model=model,
        messages=messages,
        temperature=0.2, # 0 is least creative and 1 is most creative
        max_tokens=1024,
    )
    return response.choices[0].message["content"]


def categorize_businesses(businesses, categories: set):
    prompt = "Categorize the following businesses into only the provided categories. If you cant find a category for a business in the list put the business into a category called undefined. The categories are:\n\n"
    
    for category in categories:
        prompt += f"- {category}\n"

    prompt += "\nHere are the businesses to categorize:\n\n"
    for business in businesses:
        prompt += f"- {business}\n"
    
    prompt += "\nPlease provide the categorized businesses in the following format: '<business> - <category>'.\n"

    messages = [
        {"role": "system", "content": "You are an assistant who knows how to listen to instructions."},
        {"role": "user", "content": prompt},
    ]

    response_text = get_completion(messages).strip()

    categorized_businesses = {}
    lines = response_text.split('\n')

    for line in lines:
        if ' - ' in line:
            business, category = line.split(' - ')
            business = business.strip()
            category = category.strip()

            if category not in categorized_businesses:
                categorized_businesses[category] = [business]
            else:
                categorized_businesses[category].append(business)
                
    # Move remaining uncategorized businesses to a separate category 
#     if uncategorized_businesses:
#         categorized_businesses["Uncategorized"] = list(uncategorized_businesses)

    return categorized_businesses


# categorized_businesses = categorize_businesses(businesses_set, categories_set)

# for category, businesses in categorized_businesses.items():
#     print(f"{category}: {businesses}")


In [15]:
categorized_businesses1 = categorize_businesses(businesses1, categories_set)


In [16]:
# Find the keys in combined_dict that are not in categories_set
unexpected_keys = set(categorized_businesses1.keys()) - categories_set

# Print the unexpected keys
print("Unexpected keys:", unexpected_keys)
print("Number of unexpected keys:", len(unexpected_keys))

# Check if all keys in categories_set are in the combined_dict
missing_keys = categories_set - set(categorized_businesses1.keys())

# Print the missing keys
print("Missing keys:", missing_keys)
print("Number of missing keys:", len(missing_keys))

Unexpected keys: {'undefined'}
Number of unexpected keys: 1
Missing keys: {'roofing_contractor', 'amusement_park', 'insurance_agency', 'secondary_school', 'pharmacy', 'casino', 'mosque', 'city_hall', 'funeral_home', 'meal_takeaway', 'lawyer', 'movie_theater', 'spa', 'hair_care', 'car_wash', 'painter', 'train_station', 'meal_delivery', 'liquor_store', 'subway_station', 'department_store', 'veterinary_care', 'zoo', 'electrician', 'police', 'cemetery', 'car_rental', 'bicycle_store', 'bowling_alley', 'laundry', 'hardware_store', 'shoe_store', 'car_repair', 'cafe', 'moving_company', 'light_rail_station', 'bakery', 'travel_agency', 'dentist', 'book_store', 'florist', 'fire_station', 'campground', 'aquarium', 'doctor', 'taxi_stand', 'shopping_mall', 'movie_rental', 'hindu_temple', 'physiotherapist', 'jewelry_store', 'primary_school', 'drugstore', 'gym', 'furniture_store', 'library', 'synagogue', 'embassy', 'convenience_store', 'rv_park', 'supermarket'}
Number of missing keys: 61


In [18]:
undefined_items = {k: v for k, v in categorized_businesses1.items() if k == 'undefined'}

undefined_items = {}
for k, v in categorized_businesses1.items():
    if k == 'undefined':
        undefined_items[k] = v



In [19]:
undefined_items

{'undefined': ['Aromatherapy service',
  'Land surveyor',
  'Speech pathologist',
  'Meditation center',
  'Concrete contractor',
  'ATV rental service',
  'Food production',
  'Ear piercing service',
  'Amateur theater',
  'Sweets and dessert buffet',
  'Transcription service',
  'Volleyball club',
  'Immigration & naturalization service',
  'Ballroom',
  'Honey farm',
  'Beverage distributor',
  'Recycling drop-off location',
  'Mill',
  'Farm bureau',
  'Public bathroom',
  'Tile manufacturer',
  'Theatrical costume supplier',
  's office"',
  'Mailing service',
  'Telephone company',
  'African goods store',
  'Association or organization',
  'Screen store',
  'Shed builder',
  'Sewing company',
  'Optician',
  'Design agency',
  'Home cinema installation',
  'Media house',
  'Trading card store',
  'Landscape lighting designer',
  'Business networking company',
  'Community center',
  'Warehouse club',
  'Air force base',
  'Trucking company',
  'Tool & die shop',
  'Natural goods

In [20]:
categorized_businesses1

{'car_dealer': ['Used truck dealer',
  'Auto auction',
  'Auto accessories wholesaler',
  'Jaguar dealer',
  'Mitsubishi dealer'],
 'park': ['State park'],
 'undefined': ['Aromatherapy service',
  'Land surveyor',
  'Speech pathologist',
  'Meditation center',
  'Concrete contractor',
  'ATV rental service',
  'Food production',
  'Ear piercing service',
  'Amateur theater',
  'Sweets and dessert buffet',
  'Transcription service',
  'Volleyball club',
  'Immigration & naturalization service',
  'Ballroom',
  'Honey farm',
  'Beverage distributor',
  'Recycling drop-off location',
  'Mill',
  'Farm bureau',
  'Public bathroom',
  'Tile manufacturer',
  'Theatrical costume supplier',
  's office"',
  'Mailing service',
  'Telephone company',
  'African goods store',
  'Association or organization',
  'Screen store',
  'Shed builder',
  'Sewing company',
  'Optician',
  'Design agency',
  'Home cinema installation',
  'Media house',
  'Trading card store',
  'Landscape lighting designer'

In [79]:
categorized_businesses2 = categorize_businesses(businesses2, categories_set)


In [80]:
categorized_businesses3 = categorize_businesses(businesses3, categories_set)


In [81]:
categorized_businesses4 = categorize_businesses(businesses4, categories_set)


In [82]:
categorized_businesses5 = categorize_businesses(businesses5, categories_set)


In [83]:
# Combining the dictionaries
combined_dict = {}
for d in [categorized_businesses1, categorized_businesses2, categorized_businesses3, categorized_businesses4, categorized_businesses5]:
    combined_dict.update(d)

# Print the combined dictionary
print(combined_dict)

{'real_estate_agency': ['Real estate developer', 'Real estate auctioneer', 'Mortgage broker'], 'art_gallery': ['Sculpture', 'Stone carving'], 'undefined': ['Portable building manufacturer', 'Cement manufacturer', 'Iron steel contractor', 'Sheet metal contractor', 'Livestock dealer', 'Association', 'Scout hall', 'Web hosting company', 'Contractor', 'Event ticket seller', 'Civil engineer', 'Textile mill', 'Shipping company', 'Barrel supplier', 'Piano repair service', 'Alternative medicine practitioner', 'Food and beverage consultant', 'Food and beverage exporter', 'Party planner', 'Mobility equipment supplier', 'Tunnel', 'Website designer', 'Gun shop', 'Reclamation center', 'Heating contractor', 'Process server', 'Poultry farm', 'Delivery service', 'Fur service', 'Gun club', 'Environmental organization', 'Counselor', 'Nutritionist', 'Recycling collection company', 'Welder', 'Shipping service', 'Arborist and tree surgeon', 'Tool & die shop', 'Metal fabricator', 'Trophy shop', 'Interior fi

In [57]:
# To find the number of keys:
print(len(combined_dict.keys()))

# To find the number of items:
print(len(combined_dict.items()))

83
83


In [58]:
# To find the number of keys:
print(len(categorized_businesses1.keys()))

# To find the number of items:
print(len(categorized_businesses4.items()))

49
45


In [30]:
unique_keys = set()

for d in [categorized_businesses1, categorized_businesses2, categorized_businesses3, categorized_businesses4, categorized_businesses5]:
    unique_keys.update(d.keys())

print("Total unique keys:", len(unique_keys))
print("Unique keys:", unique_keys)


Total unique keys: 271
Unique keys: {'health_food_store', 'medical_clinic', 'technical_school', 'machine_shop', 'atm', 'office_supply_store', 'print_shop', 'gift_shop', 'bakery', 'glass_industry', 'physiotherapist', 'car_parts_store', 'car_rental', 'synagogue', 'music_store', 'hair_care', 'appliance_repair_service', 'sports', 'car_dealer', 'locksmith', 'web_design', 'welding_service', 'consumer_advice_center', 'garden_center', 'employment_agency', 'swimming_pool_contractor', 'store', 'hair_salon', 'nutritionist', 'park', 'restoration_service', 'home_goods_store', 'adult_entertainment_store', 'flour_mill', 'metal_fabricator', 'photography_studio', 'post_office', 'sports_complex', 'professional_services', 'casino', 'chemical_store', 'night_club', 'gas_station', 'legal_services', 'bicycle_store', 'fishing_store', 'excavation_contractor', 'transit_station', 'brewery', 'water_utility_company', 'cosmetics_store', 'animal_rescue_service', 'real_estate_agency', 'city_department_of_public_safet

In [ ]:
# def get_completion(messages, model="gpt-3.5-turbo"):
#     response = openai.ChatCompletion.create(
#         model=model,
#         messages=messages,
#         temperature=0.3,  # Increased temperature for more creativity
#         max_tokens=1024,
#     )
#     return response.choices[0].message["content"]

# def categorize_businesses(businesses: set, categories: set):
#     prompt_base = "Your task is to categorize a list of businesses into specific categories provided below. It is essential that you only use the given categories and do not create any new ones. If it initially appears that a business does not fit any category, try to think more creatively and find the most suitable category for it from the list. Each business must belong to one of the provided categories, and no new categories should be created. Use only the categories provided below:\n\n"

#     max_iterations = 5
#     iteration = 0
#     uncategorized_businesses = businesses.copy()

#     categorized_businesses = {}

#     while uncategorized_businesses and iteration < max_iterations:
#         prompt = prompt_base
#         for category in categories:
#             prompt += f"- {category}\n"

#         prompt += "\nHere is the list of businesses you need to categorize using only the provided categories:\n\n"
#         for business in uncategorized_businesses:
#             prompt += f"- {business}\n"

#         prompt += "\nWhen categorizing the businesses, please use the following format: '<business> - <category>'.\n"

#         messages = [
#             {"role": "system", "content": "You are a helpful assistant with the ability to follow instructions precisely, without making any modifications."},
#             {"role": "user", "content": prompt},
#         ]

#         response_text = get_completion(messages).strip()

#         lines = response_text.split('\n')

#         uncategorized_businesses_next = set()

#         for line in lines:
#             if ' - ' in line:
#                 business, category = line.split(' - ')
#                 business = business.strip()
#                 category = category.strip()

#                 if category in categories:
#                     if category not in categorized_businesses:
#                         categorized_businesses[category] = [business]
#                     else:
#                         categorized_businesses[category].append(business)
#                     uncategorized_businesses.discard(business)
#                 else:
#                     print(f"Unexpected category '{category}' generated for business '{business}'")
#                     uncategorized_businesses_next.add(business)

#         uncategorized_businesses = uncategorized_businesses_next
#         iteration += 1

# #     # Move remaining uncategorized businesses to a separate category (use this information for manual review)
# #     if uncategorized_businesses:
# #         categorized_businesses["Uncategorized"] = list(uncategorized_businesses)

#     return categorized_businesses


In [26]:
categorized_businesses4

{'car_repair': ['Auto radiator repair service',
  'Hydraulic repair service',
  'Mechanic'],
 'clothing_store': ['Leather wholesaler',
  's clothing store"',
  'Leather repair service',
  'Fur coat shop'],
 'store': ['Mill', 'Mining company', 'Playground equipment supplier', 'Store'],
 'talent_agency': ['Modeling agency', 'Talent agency'],
 'music_school': ['Music instructor'],
 'sports_store': ['Skateboard shop', 'Golf shop'],
 'land_surveyor': ['Land surveyor'],
 'music_store': ['Record store',
  'DJ supply store',
  'Musical instrument manufacturer',
  'Guitar store'],
 'community_center': ['Veterans center', 'Youth social services organization'],
 'art_gallery': ['Airbrushing service'],
 'spa': ['Massage spa'],
 'office': ['Call center'],
 'food_producer': ['Food producer'],
 'video_production_service': ['Video editing service'],
 'electronics_store': ['Home theater store',
  'Telecommunications equipment supplier'],
 'beauty_salon': ['Wig shop', 'Tanning salon'],
 'lodging': ['Mot

In [75]:
# Find the keys in combined_dict that are not in categories_set
unexpected_keys = set(categorized_businesses1.keys()) - categories_set

# Print the unexpected keys
print("Unexpected keys:", unexpected_keys)
print("Number of unexpected keys:", len(unexpected_keys))

# Check if all keys in categories_set are in the combined_dict
missing_keys = categories_set - set(categorized_businesses1.keys())

# Print the missing keys
print("Missing keys:", missing_keys)
print("Number of missing keys:", len(missing_keys))

Unexpected keys: {'construction', 'business', 'grocery_or_supermarket', 'music_store', 'photography', 'pool', 'theater', 'driving_school', 'counseling', 'bookstore', 'fabric_store', 'animal_shelter', 'chemical_store', 'research'}
Number of unexpected keys: 14
Missing keys: {'drugstore', 'pet_store', 'car_wash', 'meal_takeaway', 'atm', 'movie_rental', 'secondary_school', 'shopping_mall', 'bus_station', 'plumber', 'shoe_store', 'electrician', 'florist', 'synagogue', 'bank', 'locksmith', 'supermarket', 'moving_company', 'liquor_store', 'meal_delivery', 'laundry', 'funeral_home', 'spa', 'roofing_contractor', 'aquarium', 'subway_station', 'police', 'mosque', 'casino', 'embassy', 'fire_station', 'rv_park', 'cafe', 'bicycle_store', 'storage', 'train_station', 'accounting', 'local_government_office', 'bowling_alley', 'campground', 'book_store', 'cemetery', 'beauty_salon', 'airport', 'library', 'light_rail_station', 'convenience_store', 'taxi_stand', 'university', 'painter', 'movie_theater', 'd

In [48]:
# Find the keys in combined_dict that are not in categories_set
unexpected_keys = set(categorized_businesses1.keys()) - categories_set

# Print the unexpected keys
print("Unexpected keys:", unexpected_keys)
print("Number of unexpected keys:", len(unexpected_keys))

# Check if all keys in categories_set are in the combined_dict
missing_keys = categories_set - set(categorized_businesses1.keys())

# Print the missing keys
print("Missing keys:", missing_keys)
print("Number of missing keys:", len(missing_keys))

Unexpected keys: set()
Number of unexpected keys: 0
Missing keys: {'drugstore', 'primary_school', 'jewelry_store', 'atm', 'movie_rental', 'secondary_school', 'shopping_mall', 'plumber', 'shoe_store', 'electrician', 'synagogue', 'bank', 'tourist_attraction', 'locksmith', 'moving_company', 'liquor_store', 'meal_delivery', 'laundry', 'funeral_home', 'roofing_contractor', 'aquarium', 'hindu_temple', 'subway_station', 'police', 'mosque', 'casino', 'night_club', 'fire_station', 'rv_park', 'storage', 'accounting', 'bowling_alley', 'campground', 'book_store', 'cemetery', 'convenience_store', 'airport', 'library', 'light_rail_station', 'lawyer', 'taxi_stand', 'university', 'hardware_store', 'painter', 'movie_theater', 'department_store', 'pharmacy'}
Number of missing keys: 47


In [85]:
# Find the keys in combined_dict that are not in categories_set
unexpected_keys = set(combined_dict.keys()) - categories_set

# Print the unexpected keys
print("Unexpected keys:", unexpected_keys)
print("Number of unexpected keys:", len(unexpected_keys))

# Check if all keys in categories_set are in the combined_dict
missing_keys = categories_set - set(combined_dict.keys())

# Print the missing keys
print("Missing keys:", missing_keys)
print("Number of missing keys:", len(missing_keys))


Unexpected keys: {'garden', 'engineering', 'theater', 'fabric_store', 'sporting_goods_store', 'undefined', 'garbage_dump', 'music_store', 'employment_agency', 'hobby_shop', 'tourist_information_center', 'talent_agency', 'water', 'recycling', 'photography_studio', 'optician', 'butcher_shop', 'art_supply_store', 'manufacturing', 'computer_repair', 'candy_store', 'metal_workshop', 'technology', 'medical_supply', 'child_care', 'office_supply', 'discount_store'}
Number of unexpected keys: 27
Missing keys: {'car_wash', 'train_station', 'aquarium', 'subway_station', 'moving_company', 'mosque', 'light_rail_station', 'library', 'funeral_home', 'painter', 'rv_park', 'plumber', 'pharmacy'}
Number of missing keys: 13


In [84]:
import pickle

# # Set the name of the file to save the dictionary as
# filename = 'category_dict.pickle'

# # Open the file in binary mode and write the dictionary to it using the pickle.dump() function
# with open(filename, 'wb') as f:
#     pickle.dump(combined_dict, f)


In [86]:
def compare_businesses_set_and_combined_dict(businesses_set, combined_dict):
    missing_in_dict = []
    extra_in_dict = []
    
    # Check if strings in businesses_set can be found in combined_dict
    for business in businesses_set:
        if business not in combined_dict:
            missing_in_dict.append(business)
    
    # Check if there are items in combined_dict not in businesses_set
    for key in combined_dict.keys():
        if key not in businesses_set:
            extra_in_dict.append(key)
    
    return missing_in_dict, extra_in_dict

missing_in_dict, extra_in_dict = compare_businesses_set_and_combined_dict(businesses_set, combined_dict)

if missing_in_dict:
    print(f"Businesses in businesses_set not found in combined_dict: {missing_in_dict}")
else:
    print("All businesses in businesses_set are found in combined_dict")

if extra_in_dict:
    print(f"Businesses in combined_dict not found in businesses_set: {extra_in_dict}")
else:
    print("No extra businesses found in combined_dict")


Businesses in businesses_set not found in combined_dict: [nan, 'Real estate rental agency', 'Picture frame shop', 'Handyman', 'Steamed bun shop', 'Wesleyan church', 'Pediatric dentist', 'Aggregate supplier', 'Cottage', 'Korean restaurant', 'Brewpub', 'Ballroom', 'Auto electrical service', 'Title company', 'Furniture wholesaler', 'Automation company', 'Parking lot', 'Veterinarian', 'Hair care', 'Chemical wholesaler', 'Dessert shop', 'Shipping and mailing service', 'Pool cleaning service', 'Marriage or relationship counselor', 'Lawn equipment rental service', 'Lawn mower repair service', 'Glass blower', 'Occupational medical physician', 'Trucking', 'Dermatologist', 'Brick manufacturer', 'Appliance parts supplier', 'Physical therapy clinic', 'National museum', 'Floor sanding and polishing service', 'Spice store', 'Indian Muslim restaurant', 'Stained glass studio', 'Hawaiian restaurant', 'Recording studio', 'Dental insurance agency', 'Lincoln dealer', 'Driving school', 'Water purification 

In [87]:
combined_dict

{'real_estate_agency': ['Real estate developer',
  'Real estate auctioneer',
  'Mortgage broker'],
 'art_gallery': ['Sculpture', 'Stone carving'],
 'undefined': ['Portable building manufacturer',
  'Cement manufacturer',
  'Iron steel contractor',
  'Sheet metal contractor',
  'Livestock dealer',
  'Association',
  'Scout hall',
  'Web hosting company',
  'Contractor',
  'Event ticket seller',
  'Civil engineer',
  'Textile mill',
  'Shipping company',
  'Barrel supplier',
  'Piano repair service',
  'Alternative medicine practitioner',
  'Food and beverage consultant',
  'Food and beverage exporter',
  'Party planner',
  'Mobility equipment supplier',
  'Tunnel',
  'Website designer',
  'Gun shop',
  'Reclamation center',
  'Heating contractor',
  'Process server',
  'Poultry farm',
  'Delivery service',
  'Fur service',
  'Gun club',
  'Environmental organization',
  'Counselor',
  'Nutritionist',
  'Recycling collection company',
  'Welder',
  'Shipping service',
  'Arborist and tre

In [65]:
categorized_businesses4

{'car_repair': ['Auto radiator repair service',
  'Electric motor repair shop',
  'Boat repair shop',
  'Hydraulic repair service'],
 'real_estate_agency': ['Office space rental agency',
  'Retail space rental agency'],
 'meal_takeaway': ['Juice shop'],
 'clothing_store': ['Leather wholesaler',
  's clothing store"',
  'Leather repair service',
  'Fur coat shop',
  'Clothing wholesaler',
  'Haute couture fashion house',
  'Leather cleaning service'],
 'store': ['Mill',
  'Mining company',
  'Playground equipment supplier',
  'Music instructor',
  'Skateboard shop',
  'Customs broker',
  'Veterans center',
  'Airbrushing service',
  'Call center',
  'Crane service',
  'Video editing service',
  'Non-governmental organization',
  'Biotechnology company',
  'Scrapbooking store',
  'Software company',
  'Building',
  'Photographer',
  'Store',
  'Water testing service',
  'Trade fair construction company',
  'Electric utility manufacturer',
  'Blueprint service',
  'Feed manufacturer',
  '

In [ ]:
# dict1 = {'accounting': ['Tax preparation service'], 'bank': ['Bank'], 'insurance_agency': ['Insurance agency'], 'locksmith': ['Locksmith'], 'roofing_contractor': ['Construction company']}
# dict2 = {'pet_store': ['Equestrian facility', 'Horse boarding stable', 'Fishing store'], 'pharmacy': ['Pharmacy'], 'hospital': ['Medical clinic', 'Medical laboratory'], 'clothing_store': ['Consignment shop']}
# dict3 = {'doctor': ['Doctor'], 'spa': ['Massage spa', 'Spa'], 'lodging': ['Apartment building'], 'physiotherapist': ['Chiropractor'], 'electronics_store': ['Cell phone store'], 'gym': ['Weight loss service'], 'art_gallery': ['Film production company']}

# # Combine the dictionaries
# combined_dict = {}
# for d in [dict1, dict2, dict3]:
#     combined_dict.update(d)

# # Print the combined dictionary
# print(combined_dict)


In [ ]:
# Output
# {'accounting': ['Tax preparation service'], 'bank': ['Bank'], 'insurance_agency': ['Insurance agency'], 'locksmith': ['Locksmith'], 'roofing_contractor': ['Construction company'], 'pet_store': ['Equestrian facility', 'Horse boarding stable', 'Fishing st...}


In [ ]:
# using sets instead of lists

def get_completion(messages, model="gpt-3.5-turbo"):
    response = openai.ChatCompletion.create(
        model=model,
        messages=messages,
        temperature=0.01, # 0 is least creative and 1 is most creative
        max_tokens=1024,
    )
    return response.choices[0].message["content"]

def categorize_businesses(businesses: set, categories: set):
    prompt_base = "Your task is to categorize a list of businesses into specific categories provided below. It is essential that you only use the given categories and do not create any new ones. If it initially appears that a business does not fit any category, try to think more creatively and find the most suitable category for it from the list. Each business must belong to one of the provided categories, and no new categories should be created. Use only the categories provided below:\n\n"

    max_iterations = 5
    iteration = 0
    uncategorized_businesses = businesses.copy()

    categorized_businesses = {}

    while uncategorized_businesses and iteration < max_iterations:
        prompt = prompt_base
        for category in categories:
            prompt += f"- {category}\n"

        prompt += "\nHere is the list of businesses you need to categorize using only the provided categories:\n\n"
        for business in uncategorized_businesses:
            prompt += f"- {business}\n"

        prompt += "\nWhen categorizing the businesses, please use the following format: '<business> - <category>'.\n"

        messages = [
            {"role": "system", "content": "You are a helpful assistant with the ability to follow instructions precisely, without making any modifications."},
            {"role": "user", "content": prompt},
        ]

        response_text = get_completion(messages).strip()

        lines = response_text.split('\n')

        uncategorized_businesses = set()

        for line in lines:
            if ' - ' in line:
                business, category = line.split(' - ')
                business = business.strip()
                category = category.strip()

                if category in categories:
                    if category not in categorized_businesses:
                        categorized_businesses[category] = [business]
                    else:
                        categorized_businesses[category].append(business)
                else:
                    print(f"Unexpected category '{category}' generated for business '{business}'")
                    uncategorized_businesses.add(business)

        iteration += 1

    return categorized_businesses




# def categorize_businesses(businesses: set, categories: set):
#     prompt = "Your task is to categorize a list of businesses into specific categories provided below. It is essential that you only use the given categories and do not create any new ones. If it initially appears that a business does not fit any category, try to think more creatively and find the most suitable category for it from the list. Each business must belong to one of the provided categories, and no new categories should be created. Use only the categories provided below:\n\n"

#     for category in categories:
#             prompt += f"- {category}\n"

#     prompt += "\nHere is the list of businesses you need to categorize using only the provided categories:\n\n"
#     for business in businesses:
#         prompt += f"- {business}\n"

#     prompt += "\nWhen categorizing the businesses, please use the following format: '<business> - <category>'.\n"

#     messages = [
#         {"role": "system", "content": "You are a helpful assistant with the ability to follow instructions precisely, without making any modifications."},
#         {"role": "user", "content": prompt},
#     ]

#     response_text = get_completion(messages).strip()

#     categorized_businesses = {}
#     lines = response_text.split('\n')

#     for line in lines:
#         if ' - ' in line:
#             business, category = line.split(' - ')
#             business = business.strip()
#             category = category.strip()
            
#         if category in categories_set:
#             if category not in categorized_businesses:
#                 categorized_businesses[category] = [business]
#             else:
#                 categorized_businesses[category].append(business)
#         else:
#             print(f"Unexpected category '{category}' generated for business '{business}'")
           
            
            

# #             if category not in categorized_businesses:
# #                 categorized_businesses[category] = [business]
# #             else:
# #                 categorized_businesses[category].append(business)

#     return categorized_businesses


# categorized_businesses = categorize_businesses(businesses_set, categories_set)

# for category, businesses in categorized_businesses.items():
#     print(f"{category}: {businesses}")


In [ ]:
import pickle

# Open the pickled file in read binary mode
with open('dictionary.pickle', 'rb') as f:
    # Load the dictionary object from the file
    my_dict = pickle.load(f)

# Print the loaded dictionary
print(my_dict)


In [21]:
import pickle 

with open('category_dict.pickle', 'rb') as f:
    my_dict = pickle.load(f)

In [22]:
my_dict

{'real_estate_agency': ['Real estate developer',
  'Real estate auctioneer',
  'Mortgage broker'],
 'art_gallery': ['Sculpture', 'Stone carving'],
 'undefined': ['Portable building manufacturer',
  'Cement manufacturer',
  'Iron steel contractor',
  'Sheet metal contractor',
  'Livestock dealer',
  'Association',
  'Scout hall',
  'Web hosting company',
  'Contractor',
  'Event ticket seller',
  'Civil engineer',
  'Textile mill',
  'Shipping company',
  'Barrel supplier',
  'Piano repair service',
  'Alternative medicine practitioner',
  'Food and beverage consultant',
  'Food and beverage exporter',
  'Party planner',
  'Mobility equipment supplier',
  'Tunnel',
  'Website designer',
  'Gun shop',
  'Reclamation center',
  'Heating contractor',
  'Process server',
  'Poultry farm',
  'Delivery service',
  'Fur service',
  'Gun club',
  'Environmental organization',
  'Counselor',
  'Nutritionist',
  'Recycling collection company',
  'Welder',
  'Shipping service',
  'Arborist and tre

In [24]:
def compare_businesses_set_and_combined_dict(businesses_set, combined_dict):
    missing_in_dict = []
    extra_in_dict = []
    
    # Check if strings in businesses_set can be found in combined_dict
    for business in businesses_set:
        if business not in combined_dict:
            missing_in_dict.append(business)
    
    # Check if there are items in combined_dict not in businesses_set
    for key in combined_dict.keys():
        if key not in businesses_set:
            extra_in_dict.append(key)
    
    return missing_in_dict, extra_in_dict

missing_in_dict, extra_in_dict = compare_businesses_set_and_combined_dict(businesses_set, my_dict)

if missing_in_dict:
    print(f"Businesses in businesses_set not found in combined_dict: {missing_in_dict}")
else:
    print("All businesses in businesses_set are found in combined_dict")

if extra_in_dict:
    print(f"Businesses in combined_dict not found in businesses_set: {extra_in_dict}")
else:
    print("No extra businesses found in combined_dict")


Businesses in businesses_set not found in combined_dict: [nan, 'Used truck dealer', 'Aromatherapy service', 'State park', 'Nursing school', 'ATM', 'Night club', 'Land surveyor', 'Speech pathologist', 'Cutlery store', 'Spanish restaurant', 'Meditation center', 'Concrete contractor', 'Electronic parts supplier', 'Food production', 'ATV rental service', 'Ear piercing service', 'Hamburger restaurant', 'Amateur theater', 'Woodworking supply store', 'Sweets and dessert buffet', 'Medical clinic', 'Transcription service', 'Volleyball club', 'Grill', 'Immigration & naturalization service', 'Stores and shopping', 'Ballroom', 'Honey farm', 'Game store', 'Beverage distributor', 'Recycling drop-off location', 'Mill', 'Traditional restaurant', 'Farm bureau', 'Public bathroom', 'Bus and coach company', 'Moving and storage service', 'Club', 'Tile manufacturer', 'Theatrical costume supplier', 's office", ', 'Crêperie', 'Mailing service', 'Gas company', 'Western food', 'Telephone company', 'African good

In [23]:
businesses_set

{nan,
 'Used truck dealer',
 'Aromatherapy service',
 'State park',
 'Nursing school',
 'ATM',
 'Night club',
 'Land surveyor',
 'Speech pathologist',
 'Cutlery store',
 'Spanish restaurant',
 'Meditation center',
 'Concrete contractor',
 'Electronic parts supplier',
 'Food production',
 'ATV rental service',
 'Ear piercing service',
 'Hamburger restaurant',
 'Amateur theater',
 'Woodworking supply store',
 'Sweets and dessert buffet',
 'Medical clinic',
 'Transcription service',
 'Volleyball club',
 'Grill',
 'Immigration & naturalization service',
 'Stores and shopping',
 'Ballroom',
 'Honey farm',
 'Game store',
 'Beverage distributor',
 'Recycling drop-off location',
 'Mill',
 'Traditional restaurant',
 'Farm bureau',
 'Public bathroom',
 'Bus and coach company',
 'Moving and storage service',
 'Club',
 'Tile manufacturer',
 'Theatrical costume supplier',
 's office", ',
 'Crêperie',
 'Mailing service',
 'Gas company',
 'Western food',
 'Telephone company',
 'African goods store',


In [26]:
df.head(40)

,name,address,gmap_id,description,latitude,longitude,category,avg_rating,num_of_reviews,price,hours,MISC,state,relative_results,url,zip_code,primary_category
0,H&R Block,"H&R Block, 802 Lower Fayetteville Rd Ste C, Ne...",0x88f4c6329748b5b7:0x412418d65ff2fcb7,NaN,33.374876,-84.762020,"['Tax preparation service', 'Tax consultant']",4.8,58,NaN,"{'Thursday': '10AM–5PM', 'Friday': 'Closed', '...","{'Health & safety': ['Mask required', 'Staff w...",Open ⋅ Closes 5PM,"['0x88f4c6685cd3510f:0x154ee5bc850164a6', '0x8...",https://www.google.com/maps/place//data=!4m2!3...,30263,Tax preparation service
1,Gwinnett Community Bank,"Gwinnett Community Bank, 2775 Buford Hwy, Dulu...",0x88f5a28359cb48cd:0xacc5f8fde9522f87,NaN,34.007883,-84.133413,"['Bank', 'ATM']",3.0,2,NaN,"{'Thursday': '9AM–4PM', 'Friday': '9AM–5PM', '...","{'Service options': ['Drive-through'], 'Access...",Open ⋅ Closes 4PM,"['0x88f599224d1b5ec5:0xcf6966bb9d99d4fc', '0x8...",https://www.google.com/maps/place//data=!4m2!3...,30096,Bank
2,Corley Insurance Inc,"Corley Insurance Inc, 2855 Buford Hwy, Duluth,...",0x88f5a284e80502b3:0xfe92519aa603f4d9,NaN,34.006241,-84.136169,"['Insurance agency', 'Auto insurance agency', ...",4.1,8,NaN,"{'Thursday': '8:30AM–5:30PM', 'Friday': '8:30A...",{'Accessibility': ['Wheelchair accessible entr...,Open ⋅ Closes 5:30PM,"['0x88f5a284fb58e8cd:0x8dd0cb71c88449ee', '0x8...",https://www.google.com/maps/place//data=!4m2!3...,30096,Insurance agency
3,Kenn's Duluth Lock & Key,"Kenn's Duluth Lock & Key, 3395 Fox Street Suit...",0x88f5a262358cae1b:0x3039e83297ec5bed,NaN,33.998140,-84.144132,['Locksmith'],4.9,26,NaN,"{'Thursday': 'Open 24 hours', 'Friday': 'Open ...",NaN,Open 24 hours,"['0x88f5a2ebbbcebf19:0x6bdd4859a71bdb95', '0x8...",https://www.google.com/maps/place//data=!4m2!3...,30096,Locksmith
4,rangel services,"rangel services, 36 Reeves St, Norcross, GA 30071",0x88f5a112303230d1:0xde3124e294aba58f,NaN,33.936178,-84.202939,['Construction company'],4.4,6,NaN,"{'Thursday': '8AM–6PM', 'Friday': '8AM–6PM', '...","{'Service options': ['Online estimates', 'Onsi...",Open ⋅ Closes 6PM,NaN,https://www.google.com/maps/place//data=!4m2!3...,30071,Construction company
5,Cottage Hill Farm,"Cottage Hill Farm, 6575 Pirkle Dr, Cumming, GA...",0x88f587b353d27ab9:0x39d2be8556ce1425,NaN,34.323996,-84.128164,['Equestrian facility'],5.0,3,NaN,"{'Thursday': '10AM–7PM', 'Friday': '10AM–7PM',...",NaN,Open ⋅ Closes 7PM,"['0x88f58948c1f21925:0x56fac7b369a48fe7', '0x8...",https://www.google.com/maps/place//data=!4m2!3...,30028,Equestrian facility
6,Sumpthin' Tuitt Farm,"Sumpthin' Tuitt Farm, 5450 Millwood Rd, Cummin...",0x88f58948c1f21925:0x56fac7b369a48fe7,NaN,34.284244,-84.033861,['Horse boarding stable'],5.0,8,NaN,"{'Thursday': '9AM–8PM', 'Friday': '9AM–8PM', '...",NaN,Open ⋅ Closes 8PM,"['0x88f587ca7cb554c5:0xf07be533740c91b6', '0x8...",https://www.google.com/maps/place//data=!4m2!3...,30041,Horse boarding stable
7,Publix Pharmacy at The Shops at Laurel Springs,Publix Pharmacy at The Shops at Laurel Springs...,0x88f5996b7f3e8a71:0xb1c7b0cab8b98df1,NaN,34.084331,-84.159147,"['Pharmacy', 'Medical Center', 'Medical supply...",5.0,7,$$,"{'Thursday': '9AM–8PM', 'Friday': '9AM–8PM', '...","{'Service options': ['In-store pickup', 'In-st...",Open ⋅ Closes 8PM,"['0x88f5996b0bec625f:0xacabef2c24437705', '0x8...",https://www.google.com/maps/place//data=!4m2!3...,30024,Pharmacy
8,Infectious Disease Services,"Infectious Disease Services, 1405 Haw Creek Ci...",0x14e035bef67240b1:0x25727ee4c69246ac,NaN,34.167111,-84.123869,['Medical clinic'],4.0,4,NaN,NaN,{'Accessibility': ['Wheelchair accessible entr...,NaN,"['0x88f59ad90aaab275:0xa82cbb1ea05d15cb', '0x8...",https://www.google.com/maps/place//data=!4m2!3...,30041,Medical clinic
9,Refreshingly Country Consignments,"Refreshingly Country Consignments, 5634 Spring...",0x88f35ebaecc63ffd:0xc7db7d1d60b1f73,NaN,32.888990,-84.677843,"['Consignment shop', 'Thrift store']",5.0,4,NaN,"{'Thursday': '12–5PM', 'Friday': '12–5PM', 'Sa...","{'Serv